# Exporting wearable data from sahha's api

This notebook shows an example of exporting the extracted data from sahha's api.
We'll export it as csv, but this could be a database operation as well.

In [1]:
# import dependencies

import csv
import os
import json
import datetime
from pathlib import Path
from dotenv import load_dotenv

# sahha_api module implements some api endpoint with the requests library
# please, refer to `sahha_api.py` to check the implementation
from sahha_api import get_auth_token, get_demographic, get_device_information, get_health

In [2]:
# getting credentials from .env file

load_dotenv()

CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
PATIENT_ID = os.environ["PATIENT_ID"]

In [3]:
# getting authorization token from the api 
account_token = get_auth_token(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)

demographic_data = get_demographic(token=account_token, user_id=PATIENT_ID)
device_info = get_device_information(token=account_token, user_id=PATIENT_ID)

In [4]:
# get last {NUM_OF_DAYS} days of data

NUM_OF_DAYS=15

latest_date = datetime.date.today() - datetime.timedelta(days=1)
data_by_day = []

for i in range(NUM_OF_DAYS):
    date = latest_date - datetime.timedelta(days=i)
    date_str = date.strftime("%Y-%m-%d")

    # Chamada real da função
    health_data = get_health(token=account_token, user_id=PATIENT_ID, datetime=date_str)
    
    # Mapeia os dados por "type"
    daily_record = {
        "date": date_str, 
        "os": device_info["system"], 
        "gender": demographic_data["gender"].lower(),
        "timezone": device_info["timeZone"],
        "patient": PATIENT_ID, 
    }
    for item in health_data:
        value = item.get("value", None)
        data_type = item.get("type")
        daily_record[data_type] = value
    
    data_by_day.append(daily_record)

# Get column names as unique types from json response
all_keys = set()
for record in data_by_day:
    all_keys.update(record.keys())
all_keys.discard("date")
fieldnames = ["date"] + sorted(all_keys)


In [5]:
# write to csv
first_day = latest_date - datetime.timedelta(days=NUM_OF_DAYS-1)

DATA_DIR = Path(os.getcwd()) / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FILENAME = DATA_DIR / f"health_data_{first_day.isoformat()}_to_{latest_date.isoformat()}.csv"

with open(FILENAME, mode="w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for row in data_by_day:
        writer.writerow(row)